In [1]:
import io
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import os
from pathlib import Path

os.chdir('../..')
from src.geoadmin import (
    get_landbase_bymask, load_assessor_parcels_bygeom,
    load_laraic, merge_assessor_landbase
)

In [2]:
def load_la_neighborhoods(crs='EPSG:4326'):
    """LA Times neighborhood boundaries."""
    fp = "https://s3-us-west-2.amazonaws.com/mappingla.com/downloads/neighborhoods/la_county.json"
    nbhd = gpd.read_file(fp)
    if nbhd.crs != crs:
        nbhd = nbhd.to_crs(crs)
    return nbhd
lap_regions = load_la_neighborhoods()
aoi = lap_regions[lap_regions.name == "Century City"].copy()


In [3]:
def load_for_region_year(aoi, lb_yr, apd_yr, lar_yr):
    """Load landbase + assessor + LARIAC for one AOI / year triple.
    
    Returns (assessor_parcel, assessor_lar) or (None, None) if no data.
    assessor_parcel has LB_geom (landbase polygon) as active geometry.
    """
    landbase = get_landbase_bymask(aoi, lb_yr)
    if landbase.empty:
        return None, None

    polygon_geom = aoi.to_crs(4326).dissolve().iloc[0].geometry
    assessor = load_assessor_parcels_bygeom(polygon_geom)
    assessor = assessor.loc[assessor.RollYear == apd_yr]
    if assessor.empty:
        return None, None

    assessor = assessor.to_crs(landbase.crs)
    assessor_parcel = merge_assessor_landbase(assessor, landbase)

    lariac = load_laraic(aoi, lar_yr)
    lariac = lariac.to_crs(landbase.crs)

    if 'AIN' in lariac.columns:
        lariac['poly'] = lariac.geometry
        lariac['AIN'] = lariac['AIN'].astype(str)
        assessor_parcel['AIN'] = assessor_parcel['AIN'].astype(str)

        assessor_lar = pd.merge(lariac, assessor_parcel, on = 'AIN', how = 'inner')
    else:
        lariac['poly'] = lariac.geometry
        lariac_centroid = lariac.set_geometry(lariac.centroid)
        assessor_lar = gpd.sjoin(lariac_centroid, assessor_parcel, how='inner')

    # LARIAC has its own AIN field, so sjoin produces AIN_left / AIN_right.
    # We want the assessor AIN (right side) for grouping.
    # if 'AIN_right' in assessor_lar.columns:
    #     assessor_lar = assessor_lar.rename(columns={'AIN_right': 'AIN'})

    # set_geometry('poly') makes poly the active geom — call .area directly,
    # NOT ['geometry'].area which would hit the centroid column (area = 0)
    assessor_lar['geom_area'] = assessor_lar.set_geometry('poly').area

    return assessor_parcel, assessor_lar


def compute_metrics(assessor_parcel, assessor_lar):
    """Aggregate per-AIN metrics from parcel and LARIAC data."""
    ap = assessor_parcel.copy()
    al = assessor_lar.copy()

    sqftcols = ap.columns[ap.columns.str.contains('SQFTmain')]
    ap['assr_sqftmain'] = ap[sqftcols].sum(1)

    if 'LARIAC_BUILDINGS_2014_AREA' in al.columns:
        al['AREA'] = al['LARIAC_BUILDINGS_2014_AREA']
        
    al['N_FLOOR'] = (al.HEIGHT / 14).round()
    al['lar_floorarea'] = al.N_FLOOR * al.AREA

    ap['AIN'] = ap['AIN'].astype(str)
    al['AIN'] = al['AIN'].astype(str)


    results = {
        'usetype':  ap.groupby('AIN')['UseCodeDescChar2'].first(),
        'usetype3': ap.groupby('AIN')['UseCodeDescChar3'].first(),
        'usetype4': ap.groupby('AIN')['UseCodeDescChar4'].first(),

        'assr_sqftmain':       ap.groupby('AIN')['assr_sqftmain'].sum(),
        'assr_sqftmain_check': ap.groupby('AIN')['assr_sqftmain'].first(),

        'lar_n_structures': al.groupby('AIN').size(),
        'lar_area_list':    al.groupby('AIN')['AREA'].apply(list),
        'lar_area':         al.groupby('AIN')['AREA'].sum(),
        'lar_geomarea':     al.groupby('AIN')['geom_area'].sum(),
        'lar_area_max':     al.groupby('AIN')['AREA'].max(),
        'lar_geomarea_max': al.groupby('AIN')['geom_area'].max(),
        'lar_floorarea':    al.groupby('AIN')['lar_floorarea'].sum(),
        'lar_floorarea_max':al.groupby('AIN')['lar_floorarea'].max(),
    }
    return pd.DataFrame(results)

In [4]:
from tqdm import tqdm

YEARS = [
    (2013, 2014, 2014),
    (2017, 2017, 2017),
    (2021, 2020, 2020),
]

# Start fresh — delete any existing output file so we open with mode='w' first
OUTPUT_PATH = '/Users/adamswietek/Documents/PostDoc/HiddenHousing/data/processed/assessor_lariac.gpkg'
OUTPUT_PATH = Path(OUTPUT_PATH)

master_crs   = None
first_write  = True
total_written = 0

for area_name in tqdm(lap_regions.name, desc='Regions', position=0):
    aoi = lap_regions.loc[lap_regions.name == area_name]

    for lb_yr, apd_yr, lar_yr in tqdm(YEARS, desc=area_name[:20], position=1, leave=True):
        try:
            assessor_parcel, assessor_lar = load_for_region_year(aoi, lb_yr, apd_yr, lar_yr)
            if assessor_parcel is None:
                tqdm.write(f"  {area_name} {lb_yr}: no data, skipping")
                continue

            df = compute_metrics(assessor_parcel, assessor_lar)

            # Attach parcel polygon geometry
            geom_lookup = (
                assessor_parcel
                .drop_duplicates('AIN')
                .assign(AIN=lambda x: x['AIN'].astype(str))
                .set_index('AIN')[['LB_geom', 'LotArea']]
            )
            df = df.join(geom_lookup, how='left')
            df['area_name'] = area_name
            df['year']      = lb_yr
            df.index.name   = 'AIN'
            df = df.reset_index()

            # GPKG can't store Python lists — convert to CSV string
            if 'lar_area_list' in df.columns:
                df['lar_area_list'] = df['lar_area_list'].apply(
                    lambda v: ','.join(f'{x:.2f}' for x in v) if isinstance(v, list) else ''
                )

            if master_crs is None:
                master_crs = assessor_parcel.crs

            gdf = gpd.GeoDataFrame(df, geometry='LB_geom', crs=master_crs)

            # Write first chunk with mode='w'; append everything after
            mode = 'w' if first_write else 'a'
            gdf.to_file(OUTPUT_PATH, driver='GPKG', layer=str(apd_yr), mode=mode)
            first_write = False

            total_written += len(gdf)
            tqdm.write(f"  {area_name} {lb_yr}: {len(gdf):,} parcels → {OUTPUT_PATH.name}")

        except Exception as e:
            tqdm.write(f"  {area_name} {lb_yr}: ERROR – {e}")

        finally:
            # Release large objects immediately after each iteration
            for _var in ('assessor_parcel', 'assessor_lar', 'df', 'gdf', 'geom_lookup'):
                if _var in dir():
                    del _var

tqdm.write(f"\nDone. Total rows written: {total_written:,} → {OUTPUT_PATH}")


Regions:   0%|          | 0/272 [00:00<?, ?it/s]ERROR 1: PROJ: internal_proj_identify: /Users/adamswietek/opt/anaconda3/share/proj/proj.db lacks DATABASE.LAYOUT.VERSION.MAJOR / DATABASE.LAYOUT.VERSION.MINOR metadata. It comes from another PROJ installation.
ERROR 1: PROJ: internal_proj_identify: /Users/adamswietek/opt/anaconda3/share/proj/proj.db lacks DATABASE.LAYOUT.VERSION.MAJOR / DATABASE.LAYOUT.VERSION.MINOR metadata. It comes from another PROJ installation.
                                                
Regions:   0%|          | 0/272 [00:00<?, ?it/s]ERROR 1: PROJ: internal_proj_identify: /Users/adamswietek/opt/anaconda3/share/proj/proj.db lacks DATABASE.LAYOUT.VERSION.MAJOR / DATABASE.LAYOUT.VERSION.MINOR metadata. It comes from another PROJ installation.
ERROR 1: PROJ: internal_proj_identify: /Users/adamswietek/opt/anaconda3/share/proj/proj.db lacks DATABASE.LAYOUT.VERSION.MAJOR / DATABASE.LAYOUT.VERSION.MINOR metadata. It comes from another PROJ installation.


  Acton 2013: no data, skipping


                                                
Regions:   0%|          | 0/272 [00:01<?, ?it/s]    ERROR 1: PROJ: internal_proj_identify: /Users/adamswietek/opt/anaconda3/share/proj/proj.db lacks DATABASE.LAYOUT.VERSION.MAJOR / DATABASE.LAYOUT.VERSION.MINOR metadata. It comes from another PROJ installation.


  Acton 2017: no data, skipping


ERROR 1: PROJ: internal_proj_identify: /Users/adamswietek/opt/anaconda3/share/proj/proj.db lacks DATABASE.LAYOUT.VERSION.MAJOR / DATABASE.LAYOUT.VERSION.MINOR metadata. It comes from another PROJ installation.
                                                
Regions:   0%|          | 1/272 [00:01<08:02,  1.78s/it]

...

  Woodland Hills 2013: 21,937 parcels → assessor_lariac.gpkg


/opt/homebrew/Caskroom/miniforge/base/envs/samgeo/lib/python3.12/site-packages/pyogrio/core.py:280: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read_info(
/opt/homebrew/Caskroom/miniforge/base/envs/samgeo/lib/python3.12/site-packages/geopandas/io/file.py:521: UserWarning: More than one layer found in 'LARIAC5_Buildings_2017.gdb': 'LARIAC5_BUILDINGS_2017' (default), 'LARIAC4_BUILDINGS_DELETED_2017'. Specify layer parameter to avoid this warning.
  crs = pyogrio.read_info(path_or_bytes, layer=kwargs.get("layer")).get("crs")
/opt/homebrew/Caskroom/miniforge/base/envs/samgeo/lib/python3.12/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(
/opt/homebrew/Caskroom/miniforge/base/envs/samgeo/lib/python3.12/site-packages/pyogrio/geopandas.py:382: UserWarning:

  Woodland Hills 2017: 21,924 parcels → assessor_lariac.gpkg


/opt/homebrew/Caskroom/miniforge/base/envs/samgeo/lib/python3.12/site-packages/pyogrio/core.py:280: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read_info(
/opt/homebrew/Caskroom/miniforge/base/envs/samgeo/lib/python3.12/site-packages/geopandas/io/file.py:521: UserWarning: More than one layer found in 'LARIAC6_Buildings_2020.gdb': 'LARIAC6_BUILDINGS_2020' (default), 'LARIAC5_BUILDINGS_DELETED_2020'. Specify layer parameter to avoid this warning.
  crs = pyogrio.read_info(path_or_bytes, layer=kwargs.get("layer")).get("crs")
/opt/homebrew/Caskroom/miniforge/base/envs/samgeo/lib/python3.12/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(
/opt/homebrew/Caskroom/miniforge/base/envs/samgeo/lib/python3.12/site-packages/pyogrio/geopandas.py:382: UserWarning:

  Woodland Hills 2021: 21,922 parcels → assessor_lariac.gpkg

Done. Total rows written: 4,340,564 → /Users/adamswietek/Documents/PostDoc/HiddenHousing/data/processed/assessor_lariac.gpkg
